In [14]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
GTFS_ROOT        = Path("./../../ttc-delay-analysis/data/raw/gtfs")
OUTPUT_DIR       = GTFS_ROOT / "gtfs_merged_29_39"
ROUTES_KEEP      = {"29", "39"}          # route_short_name values to keep

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# GTFS primary keys per the official spec (gtfs.org/documentation/schedule/reference/)
# These define what makes a row UNIQUE within a file.
PRIMARY_KEYS = {
    "agency.csv":        ["agency_id"],
    "stops.csv":         ["stop_id"],
    "routes.csv":        ["route_id"],
    "trips.csv":         ["trip_id"],
    "stop_times.csv":    ["trip_id", "stop_sequence"],
    "calendar.csv":      ["service_id"],
    "calendar_dates.csv":["service_id", "date"],
    "fare_attributes.csv":["fare_id"],
    "fare_rules.csv":    ["fare_id", "route_id", "origin_id",
                          "destination_id", "contains_id"],   # composite (*)
    "shapes.csv":        ["shape_id", "shape_pt_sequence"],
    "frequencies.csv":   ["trip_id", "start_time"],
    "transfers.csv":     ["from_stop_id", "to_stop_id",
                          "from_trip_id", "to_trip_id",
                          "from_route_id", "to_route_id"],    # composite (*)
    "feed_info.csv":     [],   # only one row allowed
}

# Which files we want to filter to routes 29/39 only
# All others are either global (agency, feed_info) or derived (stops filtered via trips)
ROUTE_FILTERED_FILES = {
    "routes.csv", "trips.csv", "stop_times.csv",
    "shapes.csv", "frequencies.csv", "calendar.csv",
    "calendar_dates.csv", "transfers.csv", "fare_rules.csv",
}

print("✅ Config ready.")
print(f"   GTFS root : {GTFS_ROOT}")
print(f"   Output    : {OUTPUT_DIR}")
print(f"   Routes    : {ROUTES_KEEP}")

✅ Config ready.
   GTFS root : ../../ttc-delay-analysis/data/raw/gtfs
   Output    : ../../ttc-delay-analysis/data/raw/gtfs/gtfs_merged_29_39
   Routes    : {'39', '29'}


In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Discover all feed folders & extract their effective dates
# ─────────────────────────────────────────────────────────────────────────────
def extract_feed_date(folder_name: str) -> datetime | None:
    """Extract YYYY-MM-DD from folder name like GTFS_ttc_2024-12-16_1519."""
    for part in folder_name.split("_"):
        try:
            return datetime.strptime(part, "%Y-%m-%d")
        except ValueError:
            continue
    return None

folders = sorted(
    [f for f in GTFS_ROOT.iterdir()
     if f.is_dir() and f.name.startswith("GTFS_ttc")],
    key=lambda f: extract_feed_date(f.name) or datetime.min
)

print(f"Found {len(folders)} GTFS feed folders:\\n")
for f in folders:
    d = extract_feed_date(f.name)
    files = list(f.glob("*.csv"))
    print(f"  {f.name}  →  date={d.date() if d else 'unknown'}  |  {len(files)} CSV files")

Found 18 GTFS feed folders:\n
  GTFS_ttc_2024-12-16_1519  →  date=2024-12-16  |  13 CSV files
  GTFS_ttc_2024-12-30_1519  →  date=2024-12-30  |  13 CSV files
  GTFS_ttc_2025-02-10_1219  →  date=2025-02-10  |  13 CSV files
  GTFS_ttc_2025-02-28_1919  →  date=2025-02-28  |  13 CSV files
  GTFS_ttc_2025-03-04_1219  →  date=2025-03-04  |  13 CSV files
  GTFS_ttc_2025-03-27_1219  →  date=2025-03-27  |  13 CSV files
  GTFS_ttc_2025-05-08_1519  →  date=2025-05-08  |  13 CSV files
  GTFS_ttc_2025-06-18_1519  →  date=2025-06-18  |  13 CSV files
  GTFS_ttc_2025-07-25_1519  →  date=2025-07-25  |  13 CSV files
  GTFS_ttc_2025-08-29_1519  →  date=2025-08-29  |  13 CSV files
  GTFS_ttc_2025-09-19_1519  →  date=2025-09-19  |  13 CSV files
  GTFS_ttc_2025-10-14_1219  →  date=2025-10-14  |  13 CSV files
  GTFS_ttc_2025-11-13_1219  →  date=2025-11-13  |  13 CSV files
  GTFS_ttc_2025-11-13_1519  →  date=2025-11-13  |  13 CSV files
  GTFS_ttc_2025-11-19_1219  →  date=2025-11-19  |  13 CSV files
  GTFS_ttc

In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Audit what files exist in each folder and basic row counts
# ─────────────────────────────────────────────────────────────────────────────
all_filenames = set()
for folder in folders:
    all_filenames.update(f.name for f in folder.glob("*.csv"))

print(f"{'File':<25}", end="")
for f in folders:
    d = extract_feed_date(f.name)
    label = str(d.date()) if d else f.name[-8:]
    print(f"  {label}", end="")
print()
print("-" * (25 + len(folders) * 12))

for fname in sorted(all_filenames):
    print(f"{fname:<25}", end="")
    for folder in folders:
        fpath = folder / fname
        if fpath.exists():
            try:
                n = sum(1 for _ in open(fpath)) - 1  # fast line count minus header
                print(f"  {n:>9,}", end="")
            except Exception:
                print(f"  {'ERR':>9}", end="")
        else:
            print(f"  {'—':>9}", end="")
    print()

File                       2024-12-16  2024-12-30  2025-02-10  2025-02-28  2025-03-04  2025-03-27  2025-05-08  2025-06-18  2025-07-25  2025-08-29  2025-09-19  2025-10-14  2025-11-13  2025-11-13  2025-11-19  2025-12-04  2025-12-16  2025-12-30
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
agency.csv                         1          1          1          1          1          1          1          1          1          1          1          1          1          1          1          1          1          1
calendar.csv                       6          3          4          3          4          4          4          4          4          4          4          4          3          3          3          3          6          3
calendar_dates.csv                 5          0          7          

In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Inspect columns and a few sample rows from the first feed
# ─────────────────────────────────────────────────────────────────────────────
sample_folder = folders[0]
print(f"Sampling columns from: {sample_folder.name}\\n{'='*60}")

for fname in sorted(all_filenames):
    fpath = sample_folder / fname
    if not fpath.exists():
        print(f"\\n❌ {fname} — not in first feed")
        continue
    df = pd.read_csv(fpath, dtype=str, nrows=3)
    print(f"\\n📄 {fname}")
    print(f"   Columns ({len(df.columns)}): {list(df.columns)}")
    pk = PRIMARY_KEYS.get(fname, ["?"])
    print(f"   Primary key (per GTFS spec): {pk}")
    print(df.to_string(index=False))

Sampling columns from: GTFS_ttc_2024-12-16_1519\n============================================================
\n📄 agency.csv
   Columns (8): ['agency_id', 'agency_name', 'agency_url', 'agency_timezone', 'agency_lang', 'agency_phone', 'agency_fare_url', 'agency_email']
   Primary key (per GTFS spec): ['agency_id']
agency_id agency_name        agency_url agency_timezone agency_lang agency_phone agency_fare_url agency_email
        1         TTC http://www.ttc.ca America/Toronto          en 416-393-4636             NaN          NaN
\n📄 calendar.csv
   Columns (10): ['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']
   Primary key (per GTFS spec): ['service_id']
service_id monday tuesday wednesday thursday friday saturday sunday start_date end_date
         1      0       0         1        0      0        0      0   20241222 20250104
         2      0       0         0        0      0        0      1   20241222 20250104
 

In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: Find the actual route_id values used for routes 29 & 39
# Across ALL feeds (route_id might differ between feeds — very rare but possible)
# ─────────────────────────────────────────────────────────────────────────────
all_routes_dfs = []
for folder in folders:
    rpath = folder / "routes.csv"
    if rpath.exists():
        df = pd.read_csv(rpath, dtype=str)
        df["_feed"] = folder.name
        df["_feed_date"] = str(extract_feed_date(folder.name).date()
                                if extract_feed_date(folder.name) else "unknown")
        all_routes_dfs.append(df)

routes_all = pd.concat(all_routes_dfs, ignore_index=True)

# Show all route_short_names present
print("All unique route_short_names across all feeds:")
print(routes_all["route_short_name"].value_counts().to_string())

print("\\n\\nRoutes 29 & 39 across all feeds:")
target = routes_all[routes_all["route_short_name"].isin(ROUTES_KEEP)]
print(target[["route_id", "route_short_name", "route_long_name",
              "route_type", "_feed_date"]].to_string(index=False))

TARGET_ROUTE_IDS = set(target["route_id"].unique())
print(f"\\n✅ TARGET route_ids: {TARGET_ROUTE_IDS}")

All unique route_short_names across all feeds:
route_short_name
1      18
10     18
100    18
101    18
102    18
104    18
105    18
106    18
107    18
108    18
109    18
11     18
110    18
111    18
112    18
113    18
114    18
115    18
116    18
117    18
118    18
119    18
12     18
120    18
121    18
122    18
123    18
125    18
126    18
127    18
129    18
13     18
130    18
131    18
132    18
133    18
134    18
135    18
14     18
15     18
154    18
16     18
160    18
161    18
162    18
165    18
167    18
168    18
169    18
17     18
171    18
184    18
185    18
189    18
19     18
2      18
20     18
21     18
22     18
23     18
24     18
25     18
26     18
28     18
29     18
3      18
30     18
300    18
301    18
304    18
305    18
306    18
307    18
31     18
310    18
312    18
315    18
32     18
320    18
322    18
324    18
325    18
329    18
33     18
332    18
334    18
335    18
336    18
337    18
339    18
34     18
340    18
341    18
343   

In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: Per-file deep audit — detect conflicts, schema drifts, duplicates
# This is the most important cell. Read outputs carefully before merging.
# ─────────────────────────────────────────────────────────────────────────────

def audit_file(fname, pk_cols, filter_col=None, filter_vals=None):
    """
    Load file from all feeds, tag with feed date, detect:
    1. Schema differences (column sets differ between feeds)
    2. Rows where same primary key → IDENTICAL data (true duplicates → safe to deduplicate)
    3. Rows where same primary key → DIFFERENT data (conflicts → needs review)
    """
    dfs = []
    for folder in folders:
        fpath = folder / fname
        if not fpath.exists():
            continue
        df = pd.read_csv(fpath, dtype=str).fillna("")
        if filter_col and filter_vals and filter_col in df.columns:
            df = df[df[filter_col].isin(filter_vals)]
        feed_date = extract_feed_date(folder.name)
        df["_feed_date"] = str(feed_date.date()) if feed_date else folder.name
        df["_feed_folder"] = folder.name
        dfs.append(df)

    if not dfs:
        print(f"  ⚠️  No data found for {fname}")
        return None, None

    # Check schema consistency
    col_sets = [set(df.columns) - {"_feed_date", "_feed_folder"} for df in dfs]
    if len(set(frozenset(c) for c in col_sets)) > 1:
        print(f"  ⚠️  SCHEMA DRIFT detected in {fname}!")
        for folder, cols in zip([f.name for f in folders], col_sets):
            print(f"      {folder}: {sorted(cols)}")
    else:
        print(f"  ✅ Schema consistent across all feeds.")

    combined = pd.concat(dfs, ignore_index=True)
    data_cols = [c for c in combined.columns if not c.startswith("_")]

    if not pk_cols or not all(c in data_cols for c in pk_cols):
        print(f"  ℹ️  Primary key columns {pk_cols} not fully present. Doing full-row dedup only.")
        n_dupes = combined.duplicated(subset=data_cols).sum()
        print(f"  Total rows: {len(combined):,}  |  Full-row duplicates: {n_dupes:,}")
        return combined, data_cols

    # Rows that are 100% identical (same pk, same data)
    true_dupes = combined.duplicated(subset=data_cols, keep=False)
    print(f"  Total rows loaded: {len(combined):,}")
    print(f"  Full-row duplicates (safe to drop): {combined.duplicated(subset=data_cols).sum():,}")

    # Rows with same PK but different data content = CONFLICTS
    pk_groups = combined.groupby(pk_cols, dropna=False)
    conflicts = []
    for pk_val, group in pk_groups:
        content_cols = [c for c in data_cols if c not in pk_cols]
        unique_content = group[content_cols].drop_duplicates()
        if len(unique_content) > 1:
            conflicts.append(group)

    if conflicts:
        conflict_df = pd.concat(conflicts)
        print(f"  ⚠️  CONFLICTS (same PK, different data): {len(conflict_df):,} rows "
              f"across {conflict_df[pk_cols].drop_duplicates().shape[0]:,} unique PKs")
        print(f"  First 5 conflicting PKs:")
        print(conflict_df[pk_cols + ["_feed_date"]].head(20).to_string(index=False))
    else:
        print(f"  ✅ No conflicts detected — all same-PK rows are identical.")

    return combined, data_cols


# ── routes.csv ───────────────────────────────────────────────────────────────
print("\\n" + "="*70)
print("AUDIT: routes.csv  (filter to routes 29 & 39)")
print("="*70)
routes_combined, routes_dcols = audit_file(
    "routes.csv", ["route_id"],
    filter_col="route_short_name", filter_vals=ROUTES_KEEP
)

# ── trips.csv ────────────────────────────────────────────────────────────────
print("\\n" + "="*70)
print("AUDIT: trips.csv  (filter to target route_ids)")
print("="*70)
trips_combined, trips_dcols = audit_file(
    "trips.csv", ["trip_id"],
    filter_col="route_id", filter_vals=TARGET_ROUTE_IDS
)

# ── stop_times.csv ───────────────────────────────────────────────────────────
# stop_times is huge - we'll audit via trip_ids
print("\\n" + "="*70)
print("AUDIT: stop_times.csv  (note: filtered by trip_id after trips loaded)")
print("="*70)
# Get trip_ids from the trips we already loaded
if trips_combined is not None:
    TARGET_TRIP_IDS = set(trips_combined["trip_id"].dropna().unique())
    print(f"  Target trip_ids: {len(TARGET_TRIP_IDS):,} unique trips across all feeds")
    # Just audit schema and a sample — full load happens in merge step
    st_schemas = []
    for folder in folders:
        stpath = folder / "stop_times.csv"
        if stpath.exists():
            df_sample = pd.read_csv(stpath, dtype=str, nrows=5)
            st_schemas.append((folder.name, set(df_sample.columns)))
    schema_sets = set(frozenset(c) for _, c in st_schemas)
    if len(schema_sets) == 1:
        print(f"  ✅ stop_times.csv schema consistent. Columns: {sorted(list(schema_sets)[0])}")
    else:
        print(f"  ⚠️  SCHEMA DRIFT in stop_times.csv!")
        for name, cols in st_schemas:
            print(f"     {name}: {sorted(cols)}")

# ── calendar.csv ─────────────────────────────────────────────────────────────
print("\\n" + "="*70)
print("AUDIT: calendar.csv")
print("="*70)
cal_combined, cal_dcols = audit_file("calendar.csv", ["service_id"])

# ── calendar_dates.csv ───────────────────────────────────────────────────────
print("\\n" + "="*70)
print("AUDIT: calendar_dates.csv")
print("="*70)
caldates_combined, caldates_dcols = audit_file("calendar_dates.csv",
                                               ["service_id", "date"])

# ── stops.csv ────────────────────────────────────────────────────────────────
print("\\n" + "="*70)
print("AUDIT: stops.csv  (global file — all stops, will filter post-merge)")
print("="*70)
stops_combined, stops_dcols = audit_file("stops.csv", ["stop_id"])

# ── shapes.csv ───────────────────────────────────────────────────────────────
print("\\n" + "="*70)
print("AUDIT: shapes.csv  (filtered via shape_ids used by routes 29 & 39 trips)")
print("="*70)
shapes_combined, shapes_dcols = audit_file("shapes.csv",
                                            ["shape_id", "shape_pt_sequence"])

# ── transfers.csv ─────────────────────────────────────────────────────────────
print("\\n" + "="*70)
print("AUDIT: transfers.csv")
print("="*70)
transfers_combined, _ = audit_file("transfers.csv",
                                    ["from_stop_id","to_stop_id",
                                     "from_trip_id","to_trip_id",
                                     "from_route_id","to_route_id"])

# ── fare_attributes.csv ───────────────────────────────────────────────────────
print("\\n" + "="*70)
print("AUDIT: fare_attributes.csv")
print("="*70)
fare_attr_combined, _ = audit_file("fare_attributes.csv", ["fare_id"])

# ── feed_info.csv ─────────────────────────────────────────────────────────────
print("\\n" + "="*70)
print("AUDIT: feed_info.csv  (metadata — keep most recent feed only)")
print("="*70)
feed_info_combined, _ = audit_file("feed_info.csv", [])

# ── agency.csv ────────────────────────────────────────────────────────────────
print("\\n" + "="*70)
print("AUDIT: agency.csv  (global — keep deduplicated)")
print("="*70)
agency_combined, _ = audit_file("agency.csv", ["agency_id"])

print("\\n\\n✅ AUDIT COMPLETE. Review all ⚠️  warnings above before proceeding to merge.")

\n======================================================================
AUDIT: routes.csv  (filter to routes 29 & 39)
  ✅ Schema consistent across all feeds.
  Total rows loaded: 36
  Full-row duplicates (safe to drop): 30
  ⚠️  CONFLICTS (same PK, different data): 36 rows across 2 unique PKs
  First 5 conflicting PKs:
route_id _feed_date
      29 2024-12-16
      29 2024-12-30
      29 2025-02-10
      29 2025-02-28
      29 2025-03-04
      29 2025-03-27
      29 2025-05-08
      29 2025-06-18
      29 2025-07-25
      29 2025-08-29
      29 2025-09-19
      29 2025-10-14
      29 2025-11-13
      29 2025-11-13
      29 2025-11-19
      29 2025-12-04
      29 2025-12-16
      29 2025-12-30
      39 2024-12-16
      39 2024-12-30
\n======================================================================
AUDIT: trips.csv  (filter to target route_ids)
  ✅ Schema consistent across all feeds.
  Total rows loaded: 45,424
  Full-row duplicates (safe to drop): 7,017
  ⚠️  CONFLICTS (same PK, 

In [27]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 6: Drill into any conflicts found for routes.csv and trips.csv
# Run this cell only if audit_file reported conflicts above.
# ─────────────────────────────────────────────────────────────────────────────

def show_conflicts(combined_df, pk_cols, label=""):
    """Show side-by-side comparison of conflicting rows."""
    if combined_df is None:
        print(f"No data for {label}")
        return

    data_cols = [c for c in combined_df.columns if not c.startswith("_")]
    content_cols = [c for c in data_cols if c not in pk_cols]

    pk_groups = combined_df.groupby(pk_cols, dropna=False)
    n_conflicts = 0
    for pk_val, group in pk_groups:
        unique_content = group[content_cols].drop_duplicates()
        if len(unique_content) > 1:
            n_conflicts += 1
            print(f"\\n{'─'*60}")
            print(f"CONFLICT in {label}  |  PK={pk_val}")
            print(group[pk_cols + content_cols + ["_feed_date", "_feed_folder"]].to_string(index=False))
            if n_conflicts >= 10:
                remaining = sum(1 for _, g in pk_groups
                                if len(g[content_cols].drop_duplicates()) > 1) - 10
                print(f"\\n... and {remaining} more conflicts (showing first 10)")
                break

    if n_conflicts == 0:
        print(f"✅ No conflicts in {label}")

print("="*70)
print("CONFLICT DETAIL: routes.csv")
print("="*70)
show_conflicts(routes_combined, ["route_id"], "routes.csv")

print("\\n" + "="*70)
print("CONFLICT DETAIL: trips.csv")
print("="*70)
show_conflicts(trips_combined, ["trip_id"], "trips.csv")

print("\\n" + "="*70)
print("CONFLICT DETAIL: calendar.csv")
print("="*70)
show_conflicts(cal_combined, ["service_id"], "calendar.csv")

print("\\n" + "="*70)
print("CONFLICT DETAIL: stops.csv")
print("="*70)
show_conflicts(stops_combined, ["stop_id"], "stops.csv")

CONFLICT DETAIL: routes.csv
\n────────────────────────────────────────────────────────────
CONFLICT in routes.csv  |  PK=('29',)
route_id agency_id route_short_name route_long_name route_desc route_type route_url route_color route_text_color _feed_date             _feed_folder
      29         1               29        DUFFERIN                     3                ff0000           ffffff 2024-12-16 GTFS_ttc_2024-12-16_1519
      29         1               29        DUFFERIN                     3                ff0000           ffffff 2024-12-30 GTFS_ttc_2024-12-30_1519
      29         1               29        DUFFERIN                     3                ff0000           ffffff 2025-02-10 GTFS_ttc_2025-02-10_1219
      29         1               29        DUFFERIN                     3                ff0000           ffffff 2025-02-28 GTFS_ttc_2025-02-28_1919
      29         1               29        DUFFERIN                     3                ff0000           ffffff 2025-03-04 GT

In [28]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 7: Timeline — which feeds introduced changes for routes 29 & 39?
# This shows you visually when schedule changes happened.
# ─────────────────────────────────────────────────────────────────────────────

print("ROUTE-LEVEL CHANGES OVER TIME")
print("="*70)

for route_short in sorted(ROUTES_KEEP):
    print(f"\\nRoute {route_short}:")
    prev_hash = None
    for folder in folders:
        rpath = folder / "routes.csv"
        if not rpath.exists():
            continue
        df = pd.read_csv(rpath, dtype=str).fillna("")
        row = df[df["route_short_name"] == route_short]
        if row.empty:
            continue
        row_hash = hash(tuple(row.iloc[0].values))
        feed_date = extract_feed_date(folder.name)
        changed = "🔄 CHANGED" if (prev_hash is not None and row_hash != prev_hash) else "  same    "
        print(f"  {str(feed_date.date()) if feed_date else '?':12}  {changed}  "
              f"route_id={row['route_id'].values[0]}")
        prev_hash = row_hash

print("\\n\\nTRIP COUNT CHANGES OVER TIME (by route)")
print("="*70)

trip_counts = []
for folder in folders:
    tpath = folder / "trips.csv"
    if not tpath.exists():
        continue
    rpath = folder / "routes.csv"
    if not rpath.exists():
        continue
    trips_df  = pd.read_csv(tpath, dtype=str)
    routes_df = pd.read_csv(rpath, dtype=str)
    merged = trips_df.merge(routes_df[["route_id","route_short_name"]],
                             on="route_id", how="left")
    feed_date = extract_feed_date(folder.name)
    for route_short in sorted(ROUTES_KEEP):
        count = merged[merged["route_short_name"] == route_short].shape[0]
        trip_counts.append({
            "feed_date": str(feed_date.date()) if feed_date else "?",
            "route": route_short,
            "trip_count": count
        })

tc_df = pd.DataFrame(trip_counts)
print(tc_df.groupby(["feed_date","route"])["trip_count"].sum()
          .unstack("route")
          .fillna(0)
          .astype(int)
          .to_string())

print("\\n\\nCALENDAR DATE RANGES PER FEED")
print("="*70)
for folder in folders:
    cpath = folder / "calendar.csv"
    if not cpath.exists():
        continue
    df = pd.read_csv(cpath, dtype=str)
    if "start_date" in df.columns and "end_date" in df.columns:
        feed_date = extract_feed_date(folder.name)
        print(f"  {str(feed_date.date()) if feed_date else '?':12}  "
              f"service_ids: {df['service_id'].nunique():3d}  "
              f"date range: {df['start_date'].min()} → {df['end_date'].max()}")

ROUTE-LEVEL CHANGES OVER TIME
\nRoute 29:
  2024-12-16      same      route_id=29
  2024-12-30      same      route_id=29
  2025-02-10      same      route_id=29
  2025-02-28      same      route_id=29
  2025-03-04      same      route_id=29
  2025-03-27      same      route_id=29
  2025-05-08      same      route_id=29
  2025-06-18      same      route_id=29
  2025-07-25      same      route_id=29
  2025-08-29    🔄 CHANGED  route_id=29
  2025-09-19      same      route_id=29
  2025-10-14      same      route_id=29
  2025-11-13    🔄 CHANGED  route_id=29
  2025-11-13      same      route_id=29
  2025-11-19      same      route_id=29
  2025-12-04      same      route_id=29
  2025-12-16      same      route_id=29
  2025-12-30      same      route_id=29
\nRoute 39:
  2024-12-16      same      route_id=39
  2024-12-30      same      route_id=39
  2025-02-10      same      route_id=39
  2025-02-28      same      route_id=39
  2025-03-04      same      route_id=39
  2025-03-27      same      

In [29]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 8: Pre-merge checklist — YOU must review this before running Cell 10
# ─────────────────────────────────────────────────────────────────────────────

print("""
╔══════════════════════════════════════════════════════════════════╗
║              PRE-MERGE CHECKLIST — READ CAREFULLY                ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  The merge strategy below works as follows:                      ║
║                                                                  ║
║  1. routes.csv  → Full-row dedup on data columns.                ║
║     If same route_id appears with different data across feeds,   ║
║     ALL versions are kept, tagged with _feed_date.               ║
║     → You can filter by _feed_date later for analysis.           ║
║                                                                  ║
║  2. trips.csv   → Full-row dedup. Same logic as routes.          ║
║                                                                  ║
║  3. stop_times.csv → Full-row dedup (trip_id + stop_sequence).   ║
║     Unchanged trips will deduplicate cleanly.                    ║
║     Changed schedules will appear as separate rows.              ║
║                                                                  ║
║  4. calendar.csv → Full-row dedup. Different date ranges for     ║
║     same service_id = different service periods, all kept.       ║
║                                                                  ║
║  5. calendar_dates.csv → Full-row dedup on (service_id, date).   ║
║                                                                  ║
║  6. stops.csv   → Full-row dedup on stop_id.                     ║
║     Conflicting stop attributes: ALL versions kept with tag.     ║
║                                                                  ║
║  7. shapes.csv  → Full-row dedup on (shape_id, shape_pt_seq).    ║
║                                                                  ║
║  8. transfers.csv, fare_*.csv, agency.csv, feed_info.csv         ║
║     → Full-row dedup, keep all unique rows.                      ║
║                                                                  ║
║  IMPORTANT: _feed_date column is added ONLY to files that had    ║
║  conflicts. Pure duplicates are dropped silently.                ║
║                                                                  ║
║  Set CONFIRMED = True below to proceed with the merge.           ║
╚══════════════════════════════════════════════════════════════════╝
""")

CONFIRMED = True   # ← Change to True after reviewing audit output above


╔══════════════════════════════════════════════════════════════════╗
║              PRE-MERGE CHECKLIST — READ CAREFULLY                ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  The merge strategy below works as follows:                      ║
║                                                                  ║
║  1. routes.csv  → Full-row dedup on data columns.                ║
║     If same route_id appears with different data across feeds,   ║
║     ALL versions are kept, tagged with _feed_date.               ║
║     → You can filter by _feed_date later for analysis.           ║
║                                                                  ║
║  2. trips.csv   → Full-row dedup. Same logic as routes.          ║
║                                                                  ║
║  3. stop_times.csv → Full-row dedup (trip_id + stop_sequence).   ║
║     Unchanged trips will dedupl

In [26]:
import pandas as pd
from datetime import datetime

print("CALENDAR AUDIT — RED FLAGS")
print("="*60)

for folder in folders:
    cpath = folder / "calendar.csv"
    if not cpath.exists():
        continue
    df = pd.read_csv(cpath, dtype=str)
    feed_date = extract_feed_date(folder.name)
    
    if "start_date" not in df.columns or "end_date" not in df.columns:
        continue
    
    start = pd.to_datetime(df["start_date"], format="%Y%m%d", errors="coerce").min()
    end   = pd.to_datetime(df["end_date"],   format="%Y%m%d", errors="coerce").max()
    
    # Flag 1: calendar end date is before the feed publish date (stale)
    stale = end < pd.Timestamp(feed_date) if feed_date else False
    
    # Flag 2: calendar end date already past today
    expired = end < pd.Timestamp.now()
    
    flag = ""
    if stale:   flag += " 🚨 STALE (calendar expired before feed was published)"
    if expired: flag += " ⚠️  EXPIRED (past today)"
    
    print(f"  {str(feed_date.date()):12}  {str(start.date()):12} → {str(end.date()):12}  {flag}")

print("\\n\\nTRIP COUNT VOLATILITY — flag big jumps")
print("="*60)
tc_pivot = (tc_df.groupby(["feed_date","route"])["trip_count"].sum()
               .unstack("route").fillna(0).astype(int))

for route in tc_pivot.columns:
    series = tc_pivot[route]
    pct_change = series.pct_change().abs()
    big_jumps = pct_change[pct_change > 0.20]  # >20% change feed to feed
    if not big_jumps.empty:
        print(f"\\nRoute {route} — feeds with >20% trip count change:")
        for date, pct in big_jumps.items():
            prev = series.iloc[series.index.get_loc(date) - 1]
            curr = series[date]
            print(f"  {date}:  {prev} → {curr}  ({pct*100:.0f}% change)")

CALENDAR AUDIT — RED FLAGS
  2024-12-16    2024-12-22   → 2025-01-04     ⚠️  EXPIRED (past today)
  2024-12-30    2025-01-05   → 2025-02-15     ⚠️  EXPIRED (past today)
  2025-02-10    2025-02-16   → 2025-03-29     ⚠️  EXPIRED (past today)
  2025-02-28    2024-11-17   → 2024-12-21     🚨 STALE (calendar expired before feed was published) ⚠️  EXPIRED (past today)
  2025-03-04    2025-02-16   → 2025-03-29     ⚠️  EXPIRED (past today)
  2025-03-27    2025-03-30   → 2025-05-10     ⚠️  EXPIRED (past today)
  2025-05-08    2025-05-11   → 2025-06-21     ⚠️  EXPIRED (past today)
  2025-06-18    2025-06-22   → 2025-07-26     ⚠️  EXPIRED (past today)
  2025-07-25    2025-07-27   → 2025-08-30     ⚠️  EXPIRED (past today)
  2025-08-29    2025-08-31   → 2025-10-11     ⚠️  EXPIRED (past today)
  2025-09-19    2025-08-31   → 2025-10-11     ⚠️  EXPIRED (past today)
  2025-10-14    2025-10-12   → 2025-11-15     ⚠️  EXPIRED (past today)
  2025-11-13    2025-11-16   → 2025-12-20     ⚠️  EXPIRED (past toda

In [37]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 9: MERGE — Only runs if CONFIRMED = True in Cell 9
# ─────────────────────────────────────────────────────────────────────────────
assert CONFIRMED, "Set CONFIRMED = True in Cell 9 after reviewing the audit."

def load_filtered(fname, filter_col=None, filter_vals=None, usecols=None):
    """Load file from ALL feeds, optionally filter rows, tag with feed date."""
    dfs = []
    for folder in folders:
        fpath = folder / fname
        if not fpath.exists():
            continue
        df = pd.read_csv(fpath, dtype=str).fillna("")
        if usecols:
            df = df[[c for c in usecols if c in df.columns]]
        if filter_col and filter_vals and filter_col in df.columns:
            df = df[df[filter_col].isin(filter_vals)]
        feed_date = extract_feed_date(folder.name)
        df["_feed_date"] = str(feed_date.date()) if feed_date else folder.name
        dfs.append(df)
    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)


def smart_merge(combined: pd.DataFrame, pk_cols: list[str]) -> pd.DataFrame:
    """
    Deduplicate combined dataframe.
    - Rows identical across all columns (incl. data): drop duplicates, keep first.
    - Rows with same PK but different data: keep ALL, preserve _feed_date tag.
    - If no PK given: full-row dedup on data columns only.
    """
    data_cols = [c for c in combined.columns if not c.startswith("_")]

    # Always drop exact full-row duplicates (same data, same feed_date)
    combined = combined.drop_duplicates(subset=data_cols + ["_feed_date"])

    if not pk_cols or not all(c in data_cols for c in pk_cols):
        # No PK: just deduplicate on data
        return combined.drop_duplicates(subset=data_cols).reset_index(drop=True)

    # Check for conflicts (same PK, different data)
    content_cols = [c for c in data_cols if c not in pk_cols]
    has_conflict = combined.duplicated(subset=pk_cols, keep=False)

    # Rows with no conflicts: dedup on full data row
    no_conflict = combined[~has_conflict].drop_duplicates(subset=data_cols)

    # Rows with conflicts: keep all, sorted by feed_date (newest first)
    conflict = combined[has_conflict].sort_values("_feed_date", ascending=False)

    result = pd.concat([no_conflict, conflict], ignore_index=True)

    # For non-conflicted rows, _feed_date is noise — clear it for cleanliness
    # (keep it only where there are conflicts so you can trace origin)
    non_conflict_mask = ~result.duplicated(subset=pk_cols, keep=False)
    result.loc[non_conflict_mask, "_feed_date"] = ""

    return result.reset_index(drop=True)


# ─────────────────────────────────────────────────────────────────────────────
# 1. routes.csv
# ─────────────────────────────────────────────────────────────────────────────
print("Merging routes.csv...")
routes_raw = load_filtered("routes.csv",
                            filter_col="route_short_name", filter_vals=ROUTES_KEEP)
routes_merged = smart_merge(routes_raw, ["route_id"])
routes_merged.to_csv(OUTPUT_DIR / "routes.csv", index=False)
print(f"  ✅ routes.csv: {len(routes_merged):,} rows  "
      f"(from {len(routes_raw):,} across all feeds)")

# ─────────────────────────────────────────────────────────────────────────────
# 2. trips.csv  (filtered by our route_ids)
# ─────────────────────────────────────────────────────────────────────────────
print("Merging trips.csv...")
trips_raw = load_filtered("trips.csv",
                           filter_col="route_id", filter_vals=TARGET_ROUTE_IDS)
trips_merged = smart_merge(trips_raw, ["trip_id"])
trips_merged.to_csv(OUTPUT_DIR / "trips.csv", index=False)

MERGED_TRIP_IDS   = set(trips_merged["trip_id"].unique())
MERGED_SERVICE_IDS = set(trips_merged["service_id"].unique())
MERGED_SHAPE_IDS = set(trips_merged["shape_id"].dropna().unique()) if "shape_id" in trips_merged.columns else set()

print(f"  ✅ trips.csv: {len(trips_merged):,} rows")
print(f"     service_ids referenced: {len(MERGED_SERVICE_IDS):,}")
print(f"     shape_ids  referenced: {len(MERGED_SHAPE_IDS):,}")

# ─────────────────────────────────────────────────────────────────────────────
# 3. stop_times.csv  (filtered by trip_id — largest file, load feed by feed)
# ─────────────────────────────────────────────────────────────────────────────
print("Merging stop_times.csv  (may take a moment)...")
st_chunks = []
for folder in folders:
    stpath = folder / "stop_times.csv"
    if not stpath.exists():
        continue
    feed_date = extract_feed_date(folder.name)
    df = pd.read_csv(stpath, dtype=str).fillna("")
    df = df[df["trip_id"].isin(MERGED_TRIP_IDS)]
    df["_feed_date"] = str(feed_date.date()) if feed_date else folder.name
    st_chunks.append(df)

stop_times_raw = pd.concat(st_chunks, ignore_index=True) if st_chunks else pd.DataFrame()
stop_times_merged = smart_merge(stop_times_raw, ["trip_id", "stop_sequence"])
stop_times_merged.to_csv(OUTPUT_DIR / "stop_times.csv", index=False)

MERGED_STOP_IDS_FROM_ST = set(stop_times_merged["stop_id"].unique()) if "stop_id" in stop_times_merged.columns else set()

print(f"  ✅ stop_times.csv: {len(stop_times_merged):,} rows")

# ─────────────────────────────────────────────────────────────────────────────
# 4. calendar.csv  (only service_ids used by our trips)
# ─────────────────────────────────────────────────────────────────────────────
print("Merging calendar.csv...")
cal_raw = load_filtered("calendar.csv",
                         filter_col="service_id", filter_vals=MERGED_SERVICE_IDS)
cal_merged = smart_merge(cal_raw, ["service_id"])
cal_merged.to_csv(OUTPUT_DIR / "calendar.csv", index=False)
print(f"  ✅ calendar.csv: {len(cal_merged):,} rows")

# ─────────────────────────────────────────────────────────────────────────────
# 5. calendar_dates.csv  (only service_ids used by our trips)
# ─────────────────────────────────────────────────────────────────────────────
print("Merging calendar_dates.csv...")
caldates_raw = load_filtered("calendar_dates.csv",
                              filter_col="service_id", filter_vals=MERGED_SERVICE_IDS)
caldates_merged = smart_merge(caldates_raw, ["service_id", "date"])
caldates_merged.to_csv(OUTPUT_DIR / "calendar_dates.csv", index=False)
print(f"  ✅ calendar_dates.csv: {len(caldates_merged):,} rows")

# ─────────────────────────────────────────────────────────────────────────────
# 6. stops.csv  (only stops used in stop_times for our routes)
# ─────────────────────────────────────────────────────────────────────────────
print("Merging stops.csv...")
stops_raw = load_filtered("stops.csv")
stops_for_routes = stops_raw[stops_raw["stop_id"].isin(MERGED_STOP_IDS_FROM_ST)]
stops_merged = smart_merge(stops_for_routes, ["stop_id"])
stops_merged.to_csv(OUTPUT_DIR / "stops.csv", index=False)
print(f"  ✅ stops.csv: {len(stops_merged):,} rows  "
      f"(filtered from {len(stops_raw):,} total stops)")

# ─────────────────────────────────────────────────────────────────────────────
# 7. shapes.csv  (only shape_ids used by our trips)
# ─────────────────────────────────────────────────────────────────────────────
print("Merging shapes.csv...")
shapes_raw = load_filtered("shapes.csv",
                            filter_col="shape_id", filter_vals=MERGED_SHAPE_IDS)
shapes_merged = smart_merge(shapes_raw, ["shape_id", "shape_pt_sequence"])
shapes_merged.to_csv(OUTPUT_DIR / "shapes.csv", index=False)
print(f"  ✅ shapes.csv: {len(shapes_merged):,} rows")

# ─────────────────────────────────────────────────────────────────────────────
# 8. transfers.csv  (keep rows involving our stops)
# ─────────────────────────────────────────────────────────────────────────────
print("Merging transfers.csv...")
transfers_raw = load_filtered("transfers.csv")
if not transfers_raw.empty and "from_stop_id" in transfers_raw.columns:
    mask = (transfers_raw["from_stop_id"].isin(MERGED_STOP_IDS_FROM_ST) |
            transfers_raw["to_stop_id"].isin(MERGED_STOP_IDS_FROM_ST))
    transfers_filtered = transfers_raw[mask]
else:
    transfers_filtered = transfers_raw
transfers_merged = smart_merge(transfers_filtered,
                                ["from_stop_id","to_stop_id",
                                 "from_trip_id","to_trip_id",
                                 "from_route_id","to_route_id"])
transfers_merged.to_csv(OUTPUT_DIR / "transfers.csv", index=False)
print(f"  ✅ transfers.csv: {len(transfers_merged):,} rows")

# ─────────────────────────────────────────────────────────────────────────────
# 9. fare_attributes.csv  + fare_rules.csv  (global, just dedup)
# ─────────────────────────────────────────────────────────────────────────────
print("Merging fare_attributes.csv...")
fare_attr_raw    = load_filtered("fare_attributes.csv")
fare_attr_merged = smart_merge(fare_attr_raw, ["fare_id"])
fare_attr_merged.to_csv(OUTPUT_DIR / "fare_attributes.csv", index=False)
print(f"  ✅ fare_attributes.csv: {len(fare_attr_merged):,} rows")

print("Merging fare_rules.csv...")
fare_rules_raw    = load_filtered("fare_rules.csv",
                                   filter_col="route_id", filter_vals=TARGET_ROUTE_IDS)
fare_rules_merged = smart_merge(fare_rules_raw, [])   # composite PK (*)
fare_rules_merged.to_csv(OUTPUT_DIR / "fare_rules.csv", index=False)
print(f"  ✅ fare_rules.csv: {len(fare_rules_merged):,} rows")

# ─────────────────────────────────────────────────────────────────────────────
# 10. agency.csv  (global — keep all unique agencies)
# ─────────────────────────────────────────────────────────────────────────────
print("Merging agency.csv...")
agency_raw    = load_filtered("agency.csv")
agency_merged = smart_merge(agency_raw, ["agency_id"])
agency_merged.to_csv(OUTPUT_DIR / "agency.csv", index=False)
print(f"  ✅ agency.csv: {len(agency_merged):,} rows")

# ─────────────────────────────────────────────────────────────────────────────
# 11. frequencies.csv  (optional — headway-based trips, filter by trip_id)
# ─────────────────────────────────────────────────────────────────────────────
print("Merging frequencies.csv...")
freq_raw = load_filtered("frequencies.csv",
                          filter_col="trip_id", filter_vals=MERGED_TRIP_IDS)
if not freq_raw.empty:
    freq_merged = smart_merge(freq_raw, ["trip_id", "start_time"])
    freq_merged.to_csv(OUTPUT_DIR / "frequencies.csv", index=False)
    print(f"  ✅ frequencies.csv: {len(freq_merged):,} rows")
else:
    print("  ℹ️  frequencies.csv: not present or empty for routes 29/39 — skipped")

# ─────────────────────────────────────────────────────────────────────────────
# 12. feed_info.csv  (keep only the MOST RECENT feed's info)
# ─────────────────────────────────────────────────────────────────────────────
print("Writing feed_info.csv  (most recent feed only)...")
fi_path = folders[-1] / "feed_info.csv"   # last folder = most recent date
if fi_path.exists():
    fi_df = pd.read_csv(fi_path, dtype=str)
    fi_df["_feed_date"] = str(extract_feed_date(folders[-1].name).date())
    fi_df.to_csv(OUTPUT_DIR / "feed_info.csv", index=False)
    print(f"  ✅ feed_info.csv: taken from {folders[-1].name}")
else:
    print(f"  ℹ️  feed_info.csv not found in most recent feed — skipped")

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("\\n" + "="*70)
print("MERGE COMPLETE — files written to:")
print(f"  {OUTPUT_DIR}")
print("="*70)
for f in sorted(OUTPUT_DIR.glob("*.csv")):
    n = sum(1 for _ in open(f)) - 1
    print(f"  {f.name:<25}  {n:>10,} rows")

Merging routes.csv...
  ✅ routes.csv: 34 rows  (from 36 across all feeds)
Merging trips.csv...
  ✅ trips.csv: 43,442 rows
     service_ids referenced: 6
     shape_ids  referenced: 34
Merging stop_times.csv  (may take a moment)...
  ✅ stop_times.csv: 5,957,520 rows
Merging calendar.csv...
  ✅ calendar.csv: 66 rows
Merging calendar_dates.csv...
  ✅ calendar_dates.csv: 70 rows
Merging stops.csv...
  ✅ stops.csv: 154,884 rows  (filtered from 165,012 total stops)
Merging shapes.csv...
  ✅ shapes.csv: 248,428 rows
Merging transfers.csv...
  ✅ transfers.csv: 2 rows
Merging fare_attributes.csv...
  ✅ fare_attributes.csv: 34 rows
Merging fare_rules.csv...
  ✅ fare_rules.csv: 0 rows
Merging agency.csv...
  ✅ agency.csv: 18 rows
Merging frequencies.csv...
  ℹ️  frequencies.csv: not present or empty for routes 29/39 — skipped
Writing feed_info.csv  (most recent feed only)...
  ✅ feed_info.csv: taken from GTFS_ttc_2025-12-30_1219
\n==================================================================

In [41]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 11: Final summary — what is in the merged dataset
# ─────────────────────────────────────────────────────────────────────────────

# Load merged files (defines routes_m, trips_m etc. if Cell 11 was skipped)
def load_merged(fname):
    fpath = OUTPUT_DIR / fname
    if fpath.exists():
        return pd.read_csv(fpath, dtype=str).fillna("")
    return pd.DataFrame()

routes_m     = load_merged("routes.csv")
trips_m      = load_merged("trips.csv")
stop_times_m = load_merged("stop_times.csv")
stops_m      = load_merged("stops.csv")
calendar_m   = load_merged("calendar.csv")
shapes_m     = load_merged("shapes.csv")

print("MERGED DATASET SUMMARY")
print("="*70)
print(f"Output directory: {OUTPUT_DIR}\\n")

total_rows = 0
for fpath in sorted(OUTPUT_DIR.glob("*.csv")):
    df = pd.read_csv(fpath, dtype=str)
    n  = len(df)
    total_rows += n
    conflict_col = df["_feed_date"].str.strip().ne("").sum() if "_feed_date" in df.columns else 0
    note = f"  [{conflict_col:,} versioned rows]" if conflict_col > 0 else ""
    print(f"  {fpath.name:<25}  {n:>10,} rows{note}")

print(f"\\n  {'TOTAL':<25}  {total_rows:>10,} rows")

# Per-route breakdown
print("\\n\\nPER-ROUTE BREAKDOWN")
print("─"*50)
if not routes_m.empty and not trips_m.empty and not stop_times_m.empty:
    for _, route_row in routes_m.iterrows():
        rid   = route_row["route_id"]
        rname = route_row.get("route_short_name", "?")
        rlong = route_row.get("route_long_name", "")

        route_trips = trips_m[trips_m["route_id"] == rid]["trip_id"].unique()
        n_trips  = len(route_trips)
        n_st     = stop_times_m[stop_times_m["trip_id"].isin(route_trips)].shape[0]
        n_stops  = stop_times_m[stop_times_m["trip_id"].isin(route_trips)]["stop_id"].nunique()

        print(f"  Route {rname} ({rlong[:40]})")
        print(f"    route_id     : {rid}")
        print(f"    unique trips : {n_trips:,}  (incl. versioned duplicates)")
        print(f"    stop_times   : {n_st:,}")
        print(f"    unique stops : {n_stops:,}")
        if not shapes_m.empty and "shape_id" in trips_m.columns:
            n_shapes = trips_m[trips_m["route_id"] == rid]["shape_id"].nunique()
            print(f"    unique shapes: {n_shapes:,}")
        print()

# Date coverage
if not calendar_m.empty:
    cal_tmp = calendar_m.copy()
    cal_tmp["start_date"] = pd.to_datetime(cal_tmp["start_date"], format="%Y%m%d", errors="coerce")
    cal_tmp["end_date"]   = pd.to_datetime(cal_tmp["end_date"],   format="%Y%m%d", errors="coerce")
    print(f"Service date coverage : {cal_tmp['start_date'].min().date()} → {cal_tmp['end_date'].max().date()}")
    print(f"Number of feeds merged: {len(folders)}")
    print(f"Feed date range       : {str(extract_feed_date(folders[0].name).date())} → {str(extract_feed_date(folders[-1].name).date())}")

MERGED DATASET SUMMARY
Output directory: ../../ttc-delay-analysis/data/raw/gtfs/gtfs_merged_29_39\n
  agency.csv                         18 rows  [18 versioned rows]
  calendar.csv                       66 rows  [66 versioned rows]
  calendar_dates.csv                 70 rows  [70 versioned rows]
  fare_attributes.csv                34 rows  [34 versioned rows]
  fare_rules.csv                      0 rows
  feed_info.csv                       1 rows  [1 versioned rows]
  routes.csv                         34 rows  [34 versioned rows]
  shapes.csv                    248,428 rows  [248,428 versioned rows]
  stop_times.csv              5,957,520 rows  [5,957,520 versioned rows]
  stops.csv                     154,884 rows  [154,884 versioned rows]
  transfers.csv                       2 rows  [2 versioned rows]
  trips.csv                      43,442 rows  [43,442 versioned rows]
\n  TOTAL                       6,404,499 rows
\n\nPER-ROUTE BREAKDOWN
───────────────────────────────────────

In [43]:
import pandas as pd
from pathlib import Path

MERGED = Path.home() / "Desktop" / "ttc-delay-analysis" / "data" / "raw" / "gtfs" / "gtfs_merged_29_39"

# Verify it exists before loading anything
print("Path:", MERGED)
print("Exists:", MERGED.exists())
print("Files:", [f.name for f in sorted(MERGED.glob("*.csv"))])

Path: /Users/sacharavya/Desktop/ttc-delay-analysis/data/raw/gtfs/gtfs_merged_29_39
Exists: True
Files: ['agency.csv', 'calendar.csv', 'calendar_dates.csv', 'fare_attributes.csv', 'fare_rules.csv', 'feed_info.csv', 'routes.csv', 'shapes.csv', 'stop_times.csv', 'stops.csv', 'transfers.csv', 'trips.csv']


In [50]:
calendar    = pd.read_csv(MERGED / "calendar.csv",      dtype=str)
trips       = pd.read_csv(MERGED / "trips.csv",         dtype=str)
routes      = pd.read_csv(MERGED / "routes.csv",        dtype=str)
stops       = pd.read_csv(MERGED / "stops.csv",         dtype=str)
stop_times  = pd.read_csv(MERGED / "stop_times.csv",    dtype=str)
shapes      = pd.read_csv(MERGED / "shapes.csv",        dtype=str)
calendar_dates      = pd.read_csv(MERGED / "calendar_dates.csv",        dtype=str)

In [47]:
# Drop the stale feed's calendar rows
calendar_clean = calendar[calendar["_feed_date"] != "2025-02-28"]

In [52]:
calendar_clean.head(20)

,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date,_feed_date
0,1,0,0,0,0,0,1,0,20260104,20260207,2025-12-30
1,2,0,0,0,0,0,0,1,20260104,20260207,2025-12-30
2,3,1,1,1,1,1,0,0,20260104,20260207,2025-12-30
3,1,0,0,0,0,0,0,1,20251221,20260103,2025-12-16
4,2,0,0,0,0,0,1,0,20251221,20260103,2025-12-16
5,3,0,0,0,0,1,0,0,20251221,20260103,2025-12-16
6,4,0,0,0,1,0,0,0,20251221,20260103,2025-12-16
7,5,0,0,1,0,0,0,0,20251221,20260103,2025-12-16
8,6,1,1,1,0,0,0,0,20251221,20260103,2025-12-16
9,1,0,0,0,0,0,0,1,20251207,20251220,2025-12-04


In [51]:
calendar_dates.head()

,service_id,date,exception_type,_feed_date
0,3,20241224,2,NaN
1,3,20241231,1,NaN
2,4,20250102,2,NaN
3,6,20241231,2,NaN
4,6,20250102,1,NaN
